# Шаг 22. Основной расчёт по задачам

In [1]:
import pandas as pd
import numpy as np

print("=== 1. Загрузка и фильтрация данных ===")
df = pd.read_csv('df_consolidated_clean.csv', dtype={
    'release_year': 'Int64',
    'aggregate_rating': 'Int64',
    'num_figures': 'Int64',
    'decade': 'Int64',
    'years_since_release': 'Int64'
})

df_work = df[
    (df['nationality'] != 'Не указано') & 
    (df['years_from'].notna()) & 
    (df['years_to'].notna()) &
    (df['aggregate_rating'].notna())
].copy()

print(f"Исходных строк для анализа: {len(df_work)}")


print("\n=== 2. Разворачивание исторических периодов по годам ===")
df_work['year_list'] = df_work.apply(
    lambda row: list(range(int(row['years_from']), int(row['years_to']) + 1)), axis=1
)
df_expanded = df_work.explode('year_list').rename(columns={'year_list': 'year'})
df_expanded['year'] = df_expanded['year'].astype(int)
print(f"Строк после разворачивания: {len(df_expanded)}")


print("\n=== 3. Расчёт метрик редкости ===")
pivot_matrix = df_expanded.pivot_table(
    index='year', columns='nationality', values='id', aggfunc='nunique', fill_value=0
)

year_counts = pivot_matrix.sum(axis=1)
nat_counts = pivot_matrix.sum(axis=0)

year_rarity = 1 / year_counts
nat_rarity = 1 / nat_counts

year_rarity_norm = (year_rarity - year_rarity.min()) / (year_rarity.max() - year_rarity.min())
nat_rarity_norm = (nat_rarity - nat_rarity.min()) / (nat_rarity.max() - nat_rarity.min())

df_expanded['year_rarity_score'] = df_expanded['year'].map(year_rarity_norm)
df_expanded['nat_rarity_score'] = df_expanded['nationality'].map(nat_rarity_norm)
df_expanded['composite_rarity'] = df_expanded['year_rarity_score'] + df_expanded['nat_rarity_score']


# =============================================================================
# 4. РЕКОМЕНДАЦИЯ №1: "Закрыть пробелы" С УЧЁТОМ ПЕРЕКРЫТИЯ ПЕРИОДОВ
# =============================================================================
print("\n=== 4. Рекомендация №1: 'Закрыть пробелы' (с учётом перекрытия периодов) ===")

rare_kits = df_expanded.groupby('id').agg(
    header=('header', 'first'),
    manufacturer=('manufacturer', 'first'),
    nationality=('nationality', 'first'),
    years_from=('years_from', 'first'),
    years_to=('years_to', 'first'),
    aggregate_rating=('aggregate_rating', 'max'),
    max_composite_rarity=('composite_rarity', 'max')
).reset_index()

rare_kits_sorted = rare_kits.sort_values(
    ['max_composite_rarity', 'aggregate_rating'], ascending=[False, False]
).reset_index(drop=True)

def mark_redundancy(df_sorted):
    recommended = []
    redundancy_flags = []
    
    for _, row in df_sorted.iterrows():
        nat = row['nationality']
        f, t = int(row['years_from']), int(row['years_to'])
        
        is_redundant = False
        for rec_nat, rec_f, rec_t in recommended:
            if rec_nat == nat and rec_f <= f and rec_t >= t:
                is_redundant = True
                break
        
        redundancy_flags.append(is_redundant)
        if not is_redundant:
            recommended.append((nat, f, t))
    
    return redundancy_flags

rare_kits_sorted['is_redundant'] = mark_redundancy(rare_kits_sorted)
rare_kits_sorted = rare_kits_sorted.rename(columns={'id': 'ID'})

print("Топ-20 наборов для закрытия 'белых пятен' (с пометкой избыточности):")
display_cols = ['ID', 'header', 'manufacturer', 'nationality', 'years_from', 'years_to', 
                'aggregate_rating', 'max_composite_rarity', 'is_redundant']
display(rare_kits_sorted[display_cols].head(20))

print(f"\nВсего наборов в списке: {len(rare_kits_sorted)}")
print(f"Из них уникально полезных (не избыточных): {(~rare_kits_sorted['is_redundant']).sum()}")
print(f"Избыточных (период уже покрыт более широким набором): {rare_kits_sorted['is_redundant'].sum()}")


# =============================================================================
# 5. РЕКОМЕНДАЦИЯ №2: "САМЫЕ НАСЫЩЕННЫЕ ПЕРИОДЫ"
# =============================================================================
print("\n=== 5. Самые насыщенные периоды (один лучший год на национальность) ===")

hotspots = df_expanded.groupby(['nationality', 'year']).agg(
    kit_count=('id', 'nunique'),
    avg_rating=('aggregate_rating', 'mean'),
    best_rating=('aggregate_rating', 'max')
).reset_index()

hotspots_sorted = hotspots.sort_values(
    ['nationality', 'kit_count', 'avg_rating'], ascending=[True, False, False]
)

top_hotspots = hotspots_sorted.groupby('nationality').first().reset_index()

best_kit_names = []
for _, row in top_hotspots.iterrows():
    nat = row['nationality']
    year = row['year']
    niche_kits = df_expanded[(df_expanded['nationality'] == nat) & (df_expanded['year'] == year)]
    best_kit = niche_kits.loc[niche_kits['aggregate_rating'].idxmax()]
    best_kit_names.append(best_kit['header'])

top_hotspots['best_kit_header'] = best_kit_names
top_hotspots = top_hotspots.rename(columns={
    'kit_count': 'наборов_доступно',
    'avg_rating': 'средний_рейтинг',
    'best_rating': 'лучший_рейтинг'
})
top_hotspots = top_hotspots.sort_values('наборов_доступно', ascending=False)

print("Самые насыщенные периоды для каждой национальности:")
display(top_hotspots[['nationality', 'year', 'наборов_доступно', 'средний_рейтинг', 'best_kit_header']].head(20))


# =============================================================================
# 6. ВОПРОС 11 (ТАКТИЧЕСКИЙ): ЛУЧШИЕ ПРОИЗВОДИТЕЛИ ДЛЯ КАЖДОЙ НАЦИОНАЛЬНОСТИ
# =============================================================================
print("\n=== 6. Вопрос 11 (тактический): Лучшие производители для каждой национальности ===")

# 1. Агрегируем по паре (nationality, manufacturer)
manufacturer_by_nat = df_expanded.groupby(['nationality', 'manufacturer']).agg(
    kit_count=('id', 'nunique'),
    avg_rating=('aggregate_rating', 'mean')
).reset_index()

# 2. Фильтруем: оставляем только тех производителей, у которых хотя бы 3 набора по этой национальности
# (Это гарантирует, что средний рейтинг статистически обоснован, а не основан на 1-2 случайных наборах)
manufacturer_by_nat = manufacturer_by_nat[manufacturer_by_nat['kit_count'] >= 3]

# 3. Сортируем СТРОГО по убыванию среднего рейтинга внутри каждой национальности
manufacturer_by_nat_sorted = manufacturer_by_nat.sort_values(
    ['nationality', 'avg_rating'], ascending=[True, False]
)

# 4. Берём топ-3 производителя для каждой национальности 
# (Если нужен только абсолютный лидер, можно заменить .head(3) на .head(1))
top_manufacturers_per_nat = manufacturer_by_nat_sorted.groupby('nationality', observed=False).head(3).reset_index(drop=True)

print("Топ-3 производителя для каждой национальности (по среднему рейтингу, минимум 3 набора):")

# 5. Выводим на экран только ключевые колонки, акцентируя внимание на рейтинге
display_cols = ['nationality', 'manufacturer', 'avg_rating', 'kit_count']
display(top_manufacturers_per_nat[display_cols].head(30))

# Сохраняем полную версию для CSV (чтобы в файле остались все метрики для BI), 
# но на экран выводим только сфокусированную таблицу
top_manufacturers_per_nat.to_csv('top_manufacturers_by_nationality.csv', index=False, encoding='utf-8')
print("\n✅ Сохранено: 'top_manufacturers_by_nationality.csv' (полная версия для BI)")

# =============================================================================
# 7. ВОПРОС 11 (СТРАТЕГИЧЕСКИЙ): ЛУЧШИЕ ЭПОХИ ДЛЯ КАЖДОГО ПРОИЗВОДИТЕЛЯ
# =============================================================================
print("\n=== 7. Вопрос 11 (стратегический): Лучшие эпохи для каждого производителя ===")

manufacturer_era_stats = df_expanded.groupby(['manufacturer', 'era']).agg(
    kit_count=('id', 'nunique'),
    avg_rating=('aggregate_rating', 'mean')
).reset_index()

manufacturer_era_stats = manufacturer_era_stats[manufacturer_era_stats['kit_count'] >= 5]

# Сортируем: по эпохам (хронологически), затем по убыванию среднего рейтинга
era_order = ['Древний мир', 'Средневековье', 'Новое время', 'Новейшее время', 'Современность']
manufacturer_era_stats['era'] = pd.Categorical(manufacturer_era_stats['era'], categories=era_order, ordered=True)

manufacturer_era_stats_sorted = manufacturer_era_stats.sort_values(
    ['era', 'avg_rating', 'kit_count'], ascending=[True, False, False]
)

# Для каждой эпохи находим топ-3 производителя
top_manufacturers_per_era = manufacturer_era_stats_sorted.groupby('era', observed=False).head(3).reset_index(drop=True)

print("Топ-3 производителя для каждой исторической эпохи (минимум 5 наборов):")
print("(Коллекционер видит: для Древнего мира лучшие — Zvezda и Caesar)")
display(top_manufacturers_per_era[['era', 'manufacturer', 'kit_count', 'avg_rating']].head(15))


# =============================================================================
# 8. СОХРАНЕНИЕ ФИНАЛЬНЫХ ВИТРИН
# =============================================================================
print("\n=== 8. Сохранение результатов ===")

rare_kits_sorted.to_csv('final_recommendations_rare_niches.csv', index=False, encoding='utf-8')
top_hotspots.to_csv('final_recommendations_most_saturated_periods.csv', index=False, encoding='utf-8')
top_manufacturers_per_nat.to_csv('top_manufacturers_by_nationality.csv', index=False, encoding='utf-8')
top_manufacturers_per_era.to_csv('top_manufacturers_by_era.csv', index=False, encoding='utf-8')

print("✅ Сохранено: 'final_recommendations_rare_niches.csv' (закрытие пробелов)")
print("✅ Сохранено: 'final_recommendations_most_saturated_periods.csv' (самые насыщенные периоды)")
print("✅ Сохранено: 'top_manufacturers_by_nationality.csv' (лучшие производители по национальностям — тактический уровень)")
print("✅ Сохранено: 'top_manufacturers_by_era.csv' (лучшие производители по эпохам — стратегический уровень)")


print("\n" + "="*70)
print("ИТОГ ШАГА 22: Все 14 аналитических вопросов закрыты!")
print("="*70)
print("✅ Тактический уровень:")
print("   • Блок 4: Закрытие редких ниш (с учётом перекрытия периодов)")
print("   • Блок 5: Самые насыщенные периоды для каждой национальности")
print("   • Блок 6: Лучшие производители для каждой национальности (ответ на вопрос 11)")
print("\n✅ Стратегический уровень:")
print("   • Блок 7: Лучшие эпохи для каждого производителя")
print("="*70)

=== 1. Загрузка и фильтрация данных ===
Исходных строк для анализа: 2629

=== 2. Разворачивание исторических периодов по годам ===
Строк после разворачивания: 235620

=== 3. Расчёт метрик редкости ===

=== 4. Рекомендация №1: 'Закрыть пробелы' (с учётом перекрытия периодов) ===
Топ-20 наборов для закрытия 'белых пятен' (с пометкой избыточности):


,ID,header,manufacturer,nationality,years_from,years_to,aggregate_rating,max_composite_rarity,is_redundant
0,1393,Caesar Modern Militia (H063),Caesar,Somalian,2005,2005,42,1.056079,False
1,2921,Caesar Modern PRC PLA Troops (H105),Caesar,Chinese,2015,2024,49,1.000647,False
2,3032,Mars Ukrainain Defenders Set II (72143),Mars,Ukrainian,2016,2024,44,1.000118,False
3,2112,Mars Ukrainian Defenders (72138),Mars,Ukrainian,2022,2023,43,0.665178,True
4,2802,T-Model Middle East Military Man Set (TK7312),T-Model,Afghan,2000,2020,38,0.332937,False
5,2647,Caesar Mid-East Militia (Iraq & Syria) (H101),Caesar,Iraqi,1980,2018,46,0.246692,False
6,284,Orion Chechen Rebels (72002),Orion,Russian,1994,2017,45,0.218546,False
7,1785,HaT Askari (8268),HaT,Cameroonian,1914,1918,39,0.217239,False
8,911,Coates & Shine WWI German Colonial Infantry (8...,Coates & Shine,Burundi,1914,1918,30,0.217239,False
9,2211,Strelets Police Battalion (M086),Strelets,Estonian,1941,1945,39,0.200784,False



Всего наборов в списке: 2026
Из них уникально полезных (не избыточных): 381
Избыточных (период уже покрыт более широким набором): 1645

=== 5. Самые насыщенные периоды (один лучший год на национальность) ===
Самые насыщенные периоды для каждой национальности:


,nationality,year,наборов_доступно,средний_рейтинг,best_kit_header
48,German,1943,162,43.098765,Preiser Horse Drawn Light Field Howitzer (16513)
45,French,1812,87,41.689655,Revell French Mounted Guard Chasseurs (02576)
135,USA,1865,77,37.857143,IMEX American Pioneers (516)
63,Italian,80,66,40.272727,Pegasus Gladiators (7100)
19,British,1814,60,42.4,Strelets British Infantry Standing Order Arms ...
104,Russian,1944,49,42.877551,Preiser Soviet Infantrymen on a Tank (72525)
64,Japanese,1944,30,42.1,RedBox WW2 Japanese Kamikaze (72048)
133,Turkish,1601,22,38.318182,Zvezda The Janissaries (8050)
10,Austrian,1809,21,40.857143,Strelets Allied Chiefs of Staff (2) (011)
29,Confederate,1861,19,39.157895,Italeri Confederate Cavalry (6011)



=== 6. Вопрос 11 (тактический): Лучшие производители для каждой национальности ===
Топ-3 производителя для каждой национальности (по среднему рейтингу, минимум 3 набора):


,nationality,manufacturer,avg_rating,kit_count
0,Albanian,Linear-A,42.761194,3
1,Algerian,HaT,39.007584,4
2,Almoravid,HaT,44.068259,3
3,Andalusian,HaT,43.490132,4
4,Anglo-Saxon,Strelets,42.623223,4
5,Arab,Strelets,43.636986,8
6,Assyrian,Linear-A,45.915194,4
7,Assyrian,HaT,41.826866,4
8,Australian,HaT,44.230769,3
9,Australian,Strelets,41.448276,6



✅ Сохранено: 'top_manufacturers_by_nationality.csv' (полная версия для BI)

=== 7. Вопрос 11 (стратегический): Лучшие эпохи для каждого производителя ===
Топ-3 производителя для каждой исторической эпохи (минимум 5 наборов):
(Коллекционер видит: для Древнего мира лучшие — Zvezda и Caesar)


,era,manufacturer,kit_count,avg_rating
0,Древний мир,Caesar,28,46.203127
1,Древний мир,Zvezda,15,45.965562
2,Древний мир,Linear-A,60,43.243004
3,Средневековье,Valdemar,5,46.651904
4,Средневековье,Zvezda,15,45.255372
5,Средневековье,Caesar,10,44.099024
6,Новое время,Zvezda,27,46.025112
7,Новое время,Accurate,6,46.016667
8,Новое время,Revell,18,45.385714
9,Новейшее время,Pegasus,17,47.894737



=== 8. Сохранение результатов ===
✅ Сохранено: 'final_recommendations_rare_niches.csv' (закрытие пробелов)
✅ Сохранено: 'final_recommendations_most_saturated_periods.csv' (самые насыщенные периоды)
✅ Сохранено: 'top_manufacturers_by_nationality.csv' (лучшие производители по национальностям — тактический уровень)
✅ Сохранено: 'top_manufacturers_by_era.csv' (лучшие производители по эпохам — стратегический уровень)

ИТОГ ШАГА 22: Все 14 аналитических вопросов закрыты!
✅ Тактический уровень:
   • Блок 4: Закрытие редких ниш (с учётом перекрытия периодов)
   • Блок 5: Самые насыщенные периоды для каждой национальности
   • Блок 6: Лучшие производители для каждой национальности (ответ на вопрос 11)

✅ Стратегический уровень:
   • Блок 7: Лучшие эпохи для каждого производителя


## Результат шага
Выполнен главный аналитический расчёт проекта. Созданы целевые витрины данных, которые напрямую решают задачи конечных пользователей (коллекционеров и варгеймеров) на основе сложной, но прозрачной логики оценки редкости и качества. Все 14 аналитических вопросов закрыты с учётом практической логики коллекционирования. Таблицы готовы к загрузке в BI или использованию как самостоятельные рекомендации.